# 承上启下：为什么学完Tensor，必须立刻吃透 torch.autograd？

回顾上一章《PyTorch Tensor 核心详解》三大关键结论：
1. Tensor 是支持GPU、多维存储的数值容器；
2. Tensor 具备多维存储、GPU加速、深度学习算子、自动微分四大能力；
3. NumPy 致命短板：无梯度追踪、无法自动求导。

但请注意：**Tensor的"自动微分"能力并非自带的，而是由 torch.autograd 引擎赋予的。**

## 1. 深度学习训练闭环：前向 + 反向

神经网络训练完整闭环：**前向传播计算预测值 → 反向传播(BP算法)求梯度更新权重**。

- 前向传播：Tensor 的运算、卷积、激活函数足以完成；
- 反向传播BP算法：**必须依赖 torch.autograd 微分引擎**——其底层本质是链式法则自动求导。

**核心认知**：Tensor 是承载数值与状态的基础数据结构，autograd 赋予其"自动微分"能力。
缺少 autograd 时，Tensor 只是一个支持GPU的多维数组，开发者需要为每套网络手动推导梯度公式；有了 autograd，任意复杂网络（CNN、Transformer）的梯度计算全部自动化。

## 2. autograd 底层一句话原理

任意复杂损失函数 = 大量基础运算（加、乘、激活、卷积）的复合。autograd 的工作逻辑：
1. **拆分**：将复杂计算拆解为基础算子；
2. **预置导函数**：为每类基础算子内置求导公式；
3. **链式传递**：前向记录运算链路，反向沿链路自动链式求导。

全自动实现BP，无需人工推导整体损失函数的梯度公式。

> ReLU、Abs等算子在孤立临界点不可微，autograd会选取合法次梯度继续回传，不影响训练收敛。

In [ ]:
# 开篇验证：autograd 自动处理 ReLU 在负数区间的梯度（次梯度）
import torch

# 创建一个标量张量，值为 -1.0，开启梯度追踪
t = torch.tensor(-1.0, requires_grad=True)

# 对 t 应用 ReLU 激活函数
# ReLU: x>0 时输出 x，x<=0 时输出 0
relu_out = torch.relu(t)

# 调用 backward() 触发反向传播
# 底层由 torch.autograd.backward() 实现
relu_out.backward()

# 输出梯度：ReLU 在负数区间梯度为 0（次梯度）
print("ReLU梯度（t=-1）:", t.grad)  # tensor(0.)

## 本章学习目标

### 第一层：理解 autograd 的本质
1. 理解 autograd 是 PyTorch 自动化执行BP算法的微分引擎，而非Tensor的附属功能；
2. 掌握 autograd 底层三大支柱：**算子拆分 → 预置导数 → 链式法则传递**；
3. 明确缺失 autograd 时深度学习开发的核心痛点。

### 第二层：掌握 autograd 的运作载体——计算图
4. 理解动态DAG计算图是 autograd 在前向执行时自动构建的产物；
5. 掌握计算图的两类核心节点及其协作机制；
6. 能够通过 grad_fn + next_functions 观测、拆解计算图。

### 第三层：熟练操控 autograd 的工程工具
7. 掌握 requires_grad 梯度追踪开关的传播规则；
8. 区分 detach() 与 torch.no_grad() 的本质差异及工程选型；
9. 掌握计算图生命周期控制手段（retain_graph、梯度清零等）；
10. 规范区分训练/推理场景下 autograd 的启停策略。

# 一、torch.autograd 是什么？—— 微分引擎总览

## 1.1 autograd 的核心定位

torch.autograd 是 PyTorch 的**自动微分引擎**，其唯一且核心的职责是：**自动化执行反向传播算法（BP）**。

它独立于 Tensor 存在，但通过 Tensor 的三个属性与 Tensor 协作：

| Tensor属性 | 含义 | 控制权 | 在autograd中的作用 |
|-----------|------|--------|-------------------|
| requires_grad | 是否需要追踪该张量的运算历史 | 用户设置 | **信号开关**：告诉autograd"请追踪这个张量的全部运算" |
| grad_fn | 记录该张量是由什么基本算子计算得到的 | autograd写入 | **图入口**：指向生成该张量的反向节点，供引擎反向遍历 |
| grad | 存储该张量对loss的梯度（与张量同shape） | autograd写入 | **结果容器**：引擎计算完梯度后，将结果存入此处 |

> 一句话：**autograd 是大脑，Tensor 是承载数值与状态的基础数据结构**——大脑（autograd）指挥所有运算的记录、求导、传递；Tensor 负责存储数值、状态和执行基础计算。

## 1.2 autograd 的工作闭环

```
用户开启 requires_grad=True
        ↓
autograd 开始监控该张量的全部运算
        ↓
前向执行：autograd 动态构建计算图（记录每一步运算的依赖关系和求导规则）
        ↓
用户调用 .backward()
        ↓
autograd 沿计算图反向遍历，链式求导
        ↓
autograd 将梯度（与参数张量同shape的偏导张量）写入 Tensor.grad
```

**关键认知**：整个闭环中，Tensor 只负责"被记录"和"存结果"，所有调度、构图、求导、传递的逻辑全部由 autograd 引擎独立完成。

## 1.3 autograd 数学本质回顾

autograd 执行BP的数学基础是**链式法则**。以 y=(x+w)×b 为例：

前向链路：
```
x ─┐
   ├─→ s = x + w ─→ y = s × b
w ─┘
b ─────────────────┘
```

反向链路（autograd自动执行）：
```
dy/dx = (dy/ds) × (ds/dx) = b × 1 = b
dy/dw = (dy/ds) × (ds/dw) = b × 1 = b
dy/db = s
```

autograd 在前向阶段自动记录 s = x + w 和 y = s × b 的依赖关系；
反向阶段自动沿链路计算三个偏导数，全程无需人工干预。

**核心认知**：反向传播的最终目标是计算 loss 对**所有模型参数张量**的梯度。
数学上，对于标量函数 f(w)，其中 w 是一个向量/矩阵/高维张量，其梯度定义为：
```
grad f(w) = [∂f/∂w₁, ∂f/∂w₂, ..., ∂f/∂wₙ]ᵀ  （对向量 w 的每个分量求偏导）
```
在 PyTorch 中，每个参数张量都有自己的 `.grad`，存储的是该张量对 loss 的梯度（与参数张量同 shape）。
例如：`w.grad` 是一个与 w 同 shape 的张量，`(w.grad)[i][j] = ∂loss/∂w[i][j]`。
中间激活值的偏导数只是传递路径上的"过客"，用完即弃。

In [ ]:
# 演示 autograd 的核心接口
import torch

# 演示1：.backward() 计算梯度（偏导张量）
# 创建一个标量张量，值 2.0，开启梯度追踪
x = torch.tensor([2.0], requires_grad=True)

# 前向运算：y = x^2
y = x ** 2

# 调用 backward() 触发反向传播，计算 dy/dx
y.backward()

# 输出梯度：dy/dx = 2*x = 4.0
# x.grad 是与 x 同 shape 的张量，存储 ∂y/∂x 的逐元素值
print(f"backward() 计算梯度: {x.grad.item()}")  # 4.0

# 演示2：梯度清零（不涉及 no_grad）
# zero_() 是 in-place 操作，将梯度重置为零
x.grad.zero_()
print(f"zero_() 清零后: {x.grad}")  # tensor([0.])

# 演示3：no_grad() 关闭追踪（不涉及梯度清零）
# 重新创建张量，避免之前操作的影响
x = torch.tensor([2.0], requires_grad=True)

# with torch.no_grad(): 上下文管理器，临时禁用 autograd 追踪
# 在此区域内的所有运算都不会构建计算图
with torch.no_grad():
    y = x ** 2  # 此运算不被追踪

# 因为运算未被追踪，x.grad 保持为 None（没有梯度被计算）
print(f"no_grad() 下运算后 x.grad: {x.grad}")  # None

# 演示4：两者组合使用（典型训练场景：清零后推理）
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2
y.backward()  # 先计算一次梯度
print(f"backward 后梯度: {x.grad.item()}")  # 4.0

x.grad.zero_()  # 清零梯度
with torch.no_grad():
    y = x ** 2  # 推理时不追踪

# 已清零，且新运算不产生梯度
print(f"zero_() + no_grad() 后: {x.grad}")  # tensor([0.])

# 二、autograd 的运作载体——动态计算图 DAG

## 2.1 什么是计算图？

计算图是 autograd 在前向执行时**自动构建的数据结构**，用于记录张量之间的全部运算依赖关系。它是一个**有向无环图(DAG)**：

- **节点** = 运算（如加法、乘法、ReLU），每个节点内置该运算的求导规则
- **有向边** = 张量数据的流向（从输入指向输出）
- **无环** = 数据只能向前流动，不能循环依赖

```
计算图示意图（y = (x + w) × b）

   x ─────┐
         (+)────┐
   w ─────┘    (*)──→ y
   b ───────────┘
```

计算图是 autograd 执行反向传播的**完整蓝图**：

| 阶段 | 计算图的作用 |
|------|-------------|
| **前向传播** | 记录每一步运算的依赖关系（谁依赖谁） |
| **反向传播** | 从输出端沿图反向追溯，找到所有需要求梯度的参数张量，计算每个参数张量的梯度 |

**关键认知**：关闭梯度追踪（requires_grad=False）时，autograd 停止构图。计算图的存在完全取决于 autograd 是否在工作。

> 计算图是 autograd 的"工作底稿"——前向时记录，反向时查阅。一次反向完成后默认销毁。

## 2.2 计算图如何触发？—— requires_grad + 运算

**核心答案**：requires_grad=True 本身不会构建计算图，真正的触发动作是**执行张量运算**。

### 触发构图的两层条件（缺一不可）

- **条件A（开启追踪）**：运算链路中**至少有一个**输入张量的 `requires_grad=True`
- **条件B（执行运算）**：实际执行了张量运算（+、*、matmul、relu、conv2d 等）

```text
【时刻1】创建张量（仅登记，不创建任何图节点）
x = torch.tensor([2.0], requires_grad=True)
→ x.grad_fn = None（无图节点）

【时刻2】执行第一次运算（真正触发构图！）
s = x + 3
→ s.grad_fn = AddBackward0（图节点出现了！）

【时刻3】后续运算继续"生长"图
y = s * 5
→ 在已有图的基础上追加 MulBackward0 节点
```

> 一句话：**requires_grad = 向 autograd 发出的"订阅信号"；张量运算 = 真正触发 autograd"开工搭图"的指令。**

## 2.3 图如何串联？—— grad_fn + next_functions

autograd 依靠两个核心机制完成串联：

- **grad_fn**：Tensor 的属性，记录该张量是由什么基本算子计算得到的
  - 作用：从当前张量往回走一步，找到产生它的那个运算
  - 类比：**"我是谁生的？"**

- **next_functions**：反向节点实例的属性，存储**该节点的所有上游依赖**
  - 作用：从当前运算节点继续往回追溯，找到它的所有输入来源
  - 类比：**"我的父母是谁？"**

```
示例：y = (x + w) × b

y.grad_fn → MulBackward0 ---- y 问："谁生了我？" → 乘法节点
MulBackward0.next_functions → (AddBackward0, AccumulateGrad_b) ---- 乘法节点问："我依赖谁？" → 加法节点 和 b

s.grad_fn → AddBackward0 ---- s 问："谁生了我？" → 加法节点
AddBackward0.next_functions → (AccumulateGrad_x, AccumulateGrad_w) ---- 加法节点问："我依赖谁？" → x 和 w
```

In [ ]:
# 实验1：requires_grad 本身不构图，运算才触发
import torch

# 创建张量时开启梯度追踪，但没有任何运算
x = torch.tensor([2.0], requires_grad=True)

# grad_fn 为 None，说明此时还没有构建任何计算图节点
print("创建后 x.grad_fn =", x.grad_fn)  # None

# 执行第一次运算：加法，此时才真正触发构图
s = x + 3

# grad_fn 变为 AddBackward0，说明计算图节点已经创建
print("运算后 s.grad_fn =", s.grad_fn)  # AddBackward0

# 实验2：观测 grad_fn 和 next_functions 的串联
# 创建三个需要追踪的张量
x = torch.tensor([2.0], requires_grad=True)
w = torch.tensor([3.0], requires_grad=True)
b = torch.tensor([4.0], requires_grad=True)

# 前向传播：构建 y = (x + w) * b
s = x + w          # s = x + w，创建 AddBackward0 节点
y = s * b          # y = s * b，创建 MulBackward0 节点

print("\n=== 图串联观测 ===")
print(f"y.grad_fn = {y.grad_fn}")  # <MulBackward0 object at 0x...>

# next_functions 存储该节点的所有上游依赖
# 第一个元素是 AddBackward0（s 的来源），第二个元素是 AccumulateGrad（b 的梯度汇总节点）
print(f"y.grad_fn.next_functions = {y.grad_fn.next_functions}")
# 输出示例：((<AddBackward0 object at 0x...>, 0), (<AccumulateGrad object at 0x...>, 0))

## 2.4 动手观测：完整计算图构建与遍历

运行下方代码，完整观察计算图如何随运算逐步"生长"，以及如何从末端递归遍历整张图：

In [ ]:
# 观测完整计算图的构建与递归遍历
import torch

# 创建三个需要追踪的张量
x = torch.tensor([2.0], requires_grad=True)
w = torch.tensor([3.0], requires_grad=True)
b = torch.tensor([4.0], requires_grad=True)

# 逐步构建计算图
# 第1步：s = x + w，创建 AddBackward0 节点
s = x + w
# 第2步：y = s * b，创建 MulBackward0 节点
y = s * b
# 第3步：relu_y = relu(y)，创建 ReLUBackward0 节点
relu_y = torch.relu(y)

# 打印每个输出张量的 grad_fn，观察图节点的类型
print("=== 各节点 grad_fn ===")
print(f"s.grad_fn = {s.grad_fn}")          # <AddBackward0 object at 0x...>
print(f"y.grad_fn = {y.grad_fn}")          # <MulBackward0 object at 0x...>
print(f"relu_y.grad_fn = {relu_y.grad_fn}")  # <ReLUBackward0 object at 0x...>
print()

# 定义递归遍历函数：从末端节点出发，沿着 next_functions 反向追溯
# 最终会抵达所有叶子张量的 AccumulateGrad 节点
def traverse(node, indent=0):
    """递归遍历计算图，打印每个节点的类型"""
    if node is None:
        return
    # 打印当前节点的类型名称
    print("  " * indent + f"→ {type(node).__name__}")
    # 递归遍历所有上游依赖（node.next_functions 中的每个元素）
    for child, _ in node.next_functions:
        traverse(child, indent + 1)

print("=== 完整计算图反向追溯（末端→叶子）===")
# 从输出端 relu_y 的 grad_fn 开始遍历
traverse(relu_y.grad_fn)

# 输出解读：
# ReLUBackward0 → MulBackward0 → AddBackward0 → AccumulateGrad_x, AccumulateGrad_w, AccumulateGrad_b
# 最终抵达所有叶子张量的梯度汇总节点

**运行结果解读**：

1. 每一步前向运算，autograd 都在已有图上追加新节点；
2. `next_functions` 清晰展示了每个节点的上游依赖链；
3. 递归遍历从输出端一路追溯，最终抵达各叶子张量的 AccumulateGrad 节点——即梯度汇总终点。

# 三、操控 autograd —— 梯度追踪与切断

## 3.1 总开关：requires_grad 的传播规则

- 链路中**任意一个**张量 requires_grad=True → autograd 启动追踪，构建计算图；
- 链路**全部**张量 requires_grad=False → autograd 不追踪，不构图，无法执行BP。

## 3.2 叶子 vs 中间张量：梯度需要被保留在谁身上？

这是理解 autograd 梯度管理的核心。

**关键认知**：所有参数均是叶节点，通过梯度，网络要优化的是这些参数。

**数学背景**：对于标量函数 loss = f(w)，其中 w 是一个参数张量（向量/矩阵/高维张量），
梯度定义为对 w 中每个元素求偏导，结果是与 w 同 shape 的张量：
```
(w.grad)[i][j] = ∂loss/∂w[i][j]
```
多个参数张量各自有独立的 `.grad`，结构互不相同。

| 张量类型 | 梯度是否计算 | 梯度是否保留 | 原因 |
|---------|------------|------------|------|
| **模型参数**（叶子张量，`nn.Parameter`） | ✅ 必须计算 | ✅ **永久保留**在 `.grad` | 优化器（SGD/Adam）需要读取这些梯度来更新权重 |
| **中间张量**（前向计算结果，`h = x @ w1`） | ✅ 必须计算 | ❌ **立即丢弃**（不保存） | 只用于向上游传递梯度，传递完成后使命即完成 |

**训练的目标是更新模型参数张量**，而不是保留中间激活值的梯度。
反向传播的梯度计算过程是：
1. 从 loss 开始，计算对每一层的梯度（偏导张量）
2. 中间张量的梯度（如 ∂loss/∂h）被用来计算上一层参数张量的梯度（如 ∂loss/∂w₁）
3. 中间张量的梯度用完即弃，只保留参数张量的梯度供优化器使用

```text
训练流程：
        中间激活 h    中间激活 h2
输入 x ──→ [Linear1] ──→ [ReLU] ──→ [Linear2] ──→ loss
              ↑                          ↑
            w1(参数张量)               w2(参数张量)
            (叶子)                     (叶子)

反向传播中梯度的计算与去向：
loss → ∂loss/∂h2 → ∂loss/∂w2（写入 w2.grad，永久保留！与 w2 同 shape）
                    → ∂loss/∂h（传给上游，用完即弃）
     → ∂loss/∂h → ∂loss/∂w1（写入 w1.grad，永久保留！与 w1 同 shape）
                    → ∂loss/∂x（到此为止，不需要了）

最终需要保留的是：w1.grad（与 w1 同 shape）和 w2.grad（与 w2 同 shape）
中间张量 h.grad 和 h2.grad 使命完成，丢弃！
```

**关键理解**：
- `nn.Parameter` 创建的参数默认 `requires_grad=True` 且是叶子张量
- 优化器（如 SGD、Adam）只读取这些参数张量的 `.grad` 来更新权重
- 中间张量的梯度只负责将 ∂loss 从输出端传递到参数端，传递完成后即丢弃
- 这正是为什么 `h.grad` 为 None 而 `w1.grad` 有值

**如需保留中间梯度**：调用 `.retain_grad()`（调试/可视化时使用，正式训练慎用，会增加显存）。

## 3.3 切断 autograd 追踪的两种方式

| 方法 | 粒度 | 对autograd的影响 | 典型场景 |
|------|------|-----------------|---------|
| `.detach()` | 单个张量 | 创建一个 `requires_grad=False` 的新张量，**切断该链路上梯度从下游向上游的传播**；被截断后，上游所有节点的梯度均为 `None` | 链式梯度截断：如 GAN 中切断判别器梯度传给生成器，或冻结特征提取器 |
| `with torch.no_grad():` | 代码块 | 区域内禁止autograd新建任何计算图节点 | 推理/验证 |

**本质区别**：
- `.detach()` = 在**单条链式路径**上**切断梯度向上游传播**：下游梯度传到 `detach()` 节点后无法继续向上游传播，**上游所有节点（包括叶子）的 `.grad` 均为 `None`**
- `no_grad()` = 临时关闭整个autograd记录系统，**区域内完全不生成任何图节点**，**该区域内的任何参数都无法被更新**

**方向示意图**（箭头方向 = 梯度传播方向，从输出到输入）：
```text
梯度传播方向（从输出到输入）：
  ... → 下游节点 → h_detached ✗ 切断 ✗ 上游节点 → x
  
  梯度从输出端一路传来：
  ✅ out 收到梯度
  ✅ h_detached（requires_grad=False）能收到梯度，但没有 grad 存储
  ❌ 梯度无法继续传到上游的 h 和 x
  ❌ x.grad = None，无法计算 ∂out/∂x
```

In [ ]:
# 实验1：中间张量梯度默认丢弃 vs 参数叶子梯度保留
import torch

# 创建参数叶子 w1（类似模型参数），开启梯度追踪
w1 = torch.tensor(2.0, requires_grad=True)

# 前向传播：h = w1 * 3（h 是中间张量，is_leaf=False）
h = w1 * 3

# y = h * 5（y 也是中间张量）
y = h * 5

# 执行反向传播
y.backward()

# w1.grad 被保留（叶子张量，存储的是 ∂y/∂w1，与 w1 同 shape——标量）
print("w1.grad (参数叶子，梯度保留):", w1.grad)  # tensor(15.)

# h.grad 为 None（中间张量，梯度被丢弃）
print("h.grad (中间张量，梯度丢弃):", h.grad)  # None
print("\n关键认知：只有参数叶子（nn.Parameter）的梯度需要保留，")
print("         中间张量的梯度使命完成后即被丢弃。\n")

# 实验2：detach() vs no_grad()
x = torch.tensor([2.0], requires_grad=True)

# h = x * 3，h 是中间张量，有 grad_fn
h = x * 3

# detach() 切断 h 的梯度链路
# h_detach 是一个新的张量，与 h 共享数据，但 requires_grad=False
h_detach = h.detach()

# 后续运算不再被追踪
out_detach = h_detach + 2

# no_grad() 临时关闭整个 autograd 追踪
with torch.no_grad():
    out_no_grad = x * 3 + 2

print(f"detach 后 out_detach.requires_grad: {out_detach.requires_grad}")  # False
print(f"no_grad 后 out_no_grad.requires_grad: {out_no_grad.requires_grad}")  # False
print()

# 实验3：梯度清零 vs no_grad 是两个独立操作
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2
y.backward()
print(f"backward 后: x.grad = {x.grad.item()}")  # 4.0

# zero_()：只清零梯度，不关闭追踪
x.grad.zero_()
print(f"zero_() 后: x.grad = {x.grad}")  # tensor([0.])

# no_grad()：只关闭追踪，不清零梯度
with torch.no_grad():
    y = x ** 2
print(f"no_grad() 后: x.grad = {x.grad}")  # tensor([0.])

# 实验4：retain_grad() 强制保留中间梯度
print("\n=== 实验4：retain_grad() 强制保留中间梯度 ===")
w = torch.tensor(2.0, requires_grad=True)
m = w * 3

# 调用 retain_grad() 强制 autograd 保留 m 的梯度
# 默认情况下中间张量的梯度会被丢弃
m.retain_grad()

y = m * 5
y.backward()

print(f"w.grad (参数叶子): {w.grad}")  # tensor(15.)
print(f"m.grad (retain_grad() 强制保留): {m.grad}")  # tensor(5.)
print("注意：正式训练中 retain_grad() 会增加显存占用，慎用。")

# 四、反向传播的遍历机制

## 4.1 什么是"遍历"？

**遍历（Traversal）** 是计算机科学中一个基础概念：**按照某种规则，访问数据结构中的每一个节点恰好一次**。

- **数据结构**：树、图、链表等
- **目标**：不重复、不遗漏地访问所有节点
- **核心问题**：以什么顺序访问？

autograd 的反向传播，本质上就是**对计算图（DAG）的一次遍历**。

## 4.2 遍历的数学目的：计算每个参数张量的梯度

**遍历是手段，求梯度是目的。**

```
遍历前：只知道输出张量 y（最终输出值），以及它的 grad_fn
遍历中：从 y 出发，沿着 next_functions 不断访问新节点
遍历结束：所有 requires_grad=True 的参数叶子张量都被访问到，
          每个参数张量的梯度（与参数同 shape 的偏导张量）被计算并写入 .grad
```

**最终目的**：找到所有模型参数张量（`nn.Parameter`，即 `requires_grad=True` 的叶子张量），
计算**每个参数张量对 loss 的梯度**（与参数张量同 shape），供优化器更新权重。

**数学上**：对于参数张量 w（shape=[m,n]），其梯度是：
```
(w.grad)[i][j] = ∂loss/∂w[i][j]，i=0..m-1，j=0..n-1
```
每个参数张量各有自己的 `.grad`，shape 与对应参数张量相同。

**关键认知**：
- 模型参数（如 `nn.Linear.weight`，shape=[out, in]）是叶子张量，其梯度需要永久保留
- 中间激活值（如 `h = x @ w1`）不是优化目标，其梯度只是传递路径上的"过客"
- 遍历过程中，只有抵达参数叶子的 AccumulateGrad 节点时，梯度（偏导张量）才被写入 `.grad` 并持久保存

形象地说就是**"顺藤摸瓜"**：
- **藤** = 计算图（记录了从参数到输出的所有运算依赖）
- **瓜** = 所有需要梯度的参数张量（requires_grad=True 的叶子张量）
- **摸** = 沿 grad_fn → next_functions 反向追溯，逐级应用链式法则
- **结果** = 每个参数张量的 `.grad` 被填上对应的梯度张量

```
顺藤摸瓜的反向追溯路径：

最终输出 y
    ↓ y.grad_fn（指向产生 y 的运算节点）
    运算节点A（如 MulBackward0）
    ↓ node.next_functions（指向 A 的所有输入来源）
    运算节点B + 参数叶子 w
    ↓ 继续追溯...
    ...
    最终抵达所有参数叶子的 AccumulateGrad 节点
    ↓
    每个 AccumulateGrad 将收到的梯度写入对应叶子.grad（与叶子同 shape）
```

## 4.3 遍历的两个核心问题

### 问题1：往哪个方向走？

计算图是有向的，边从输入指向输出：

```
前向方向（数据流）：叶子(参数) → 运算节点1 → 运算节点2 → ... → 输出 y

反向方向（梯度流）：输出 y → 运算节点N → 运算节点N-1 → ... → 叶子(参数)
```

**遍历方向 = 反方向**（从输出往参数叶子走）。

为什么？因为链式法则要求从外层向内层算：
```
∂y/∂w = ∂y/∂a · ∂a/∂b · ∂b/∂w
         ↑        ↑        ↑
       先算    再算      最后算
```

### 问题2：以什么顺序访问？

**深度优先（DFS）**。

```
假设有这样的图：
        ┌── a ──┐
        │       │
    w ──┤       ├── y
        │       │
        └── b ──┘

深度优先：从 y 出发，先沿一条路走到底（y → a → w），再回溯走另一条（y → b → w）
```

**为什么是DFS，不是BFS？**

因为每个节点的梯度计算**依赖**于其下游节点的梯度：
- 要算 `∂y/∂w`，必须先算 `∂y/∂a` 和 `∂y/∂b`
- 要算 `∂y/∂a`，必须先算 `∂y/∂y`（即 1）

DFS 天然保证这种依赖顺序：**从输出端一路走到参数叶子，再回溯处理分支**。

如果换成 BFS，会出现"节点的梯度还没算完，就被用来算上游"的问题。

## 4.4 遍历的数据结构：栈（递归）

深度优先遍历的底层实现是**栈**（LIFO）。在代码中通常表现为**递归**：

```python
def traverse(node):
    if node is None:
        return
    # 访问当前节点：计算局部梯度
    compute_gradient(node)
    # 递归访问所有上游依赖
    for child in node.next_functions:
        traverse(child)
```

**每次函数调用 = 访问一个节点 = 计算该节点对应的局部梯度。**

## 4.5 完整示例：一步步拆解遍历过程

**前向计算图**：`y = (x + w) × b`，其中 w、b 是需要求梯度的参数张量，x 是输入数据

```
        x(数据) ──┐
                  ├── (+) ── s ──┐
        w(参数) ──┘              ├── (*) ── y(最终输出)
        b(参数) ─────────────────┘
```

**数学推导**（目标是求 ∂y/∂w 和 ∂y/∂b）：

```
第一步：从输出 y 开始，∂y/∂y = 1

第二步：到达乘法节点 (*)，y = s × b
       局部偏导：∂y/∂s = b = 4，∂y/∂b = s = 5

第三步：到达加法节点 (+)，s = x + w
       局部偏导：∂s/∂w = 1

第四步：链式相乘
       ∂y/∂w = (∂y/∂s) × (∂s/∂w) = 4 × 1 = 4
       ∂y/∂b = (∂y/∂b) = 5

最终结果：
       w.grad = ∂y/∂w = 4（与 w 同 shape）
       b.grad = ∂y/∂b = 5（与 b 同 shape）
```

**autograd 遍历的完整步骤**：

```text
【第1步】从 y 出发
  入口：y.grad_fn = MulBackward0
  访问 MulBackward0（第1个被访问的节点）
  计算：∂y/∂s = b，∂y/∂b = s
  将梯度向上游传递：
    → 传给 s 的 grad_fn（AddBackward0）
    → 传给 b 的 AccumulateGrad（参数叶子 b）

【第2步】进入 AddBackward0（MulBackward0 的第一个上游依赖）
  访问 AddBackward0（第2个被访问的节点）
  接收：∂y/∂s
  计算：∂s/∂w = 1
  链式相乘：∂y/∂w = ∂y/∂s × 1
  将梯度继续向上游传递：
    → 传给 w 的 AccumulateGrad（参数叶子 w）

【第3步】处理 MulBackward0 的下一个上游依赖（b 的 AccumulateGrad）
  b 的 AccumulateGrad 已收到 ∂y/∂b
  无需继续追溯（参数叶子，无 next_functions）
  将 ∂y/∂b 写入 b.grad

【第4步】处理 w 的 AccumulateGrad
  w 的 AccumulateGrad 收到 ∂y/∂w
  将 ∂y/∂w 写入 w.grad

【遍历结束】所有参数叶子都被访问到，
          ∂y/∂w 和 ∂y/∂b 计算完毕并存入各自 .grad
```

## 4.6 每个节点的职责：计算局部梯度

每个反向节点内部**预置了对应前向运算的求导公式**，作为"局部梯度计算器"：

| 前向运算 | 反向节点的职责（已知 ∂y/∂out，求 ∂y/∂input） |
|---------|--------------------------------------------------|
| y = a + b | ∂y/∂a = ∂y/∂out，∂y/∂b = ∂y/∂out |
| y = a × b | ∂y/∂a = ∂y/∂out × b，∂y/∂b = ∂y/∂out × a |
| y = x² | ∂y/∂x = ∂y/∂out × 2x |
| y = relu(x) | x>0 时 ∂y/∂x = ∂y/∂out；x≤0 时 ∂y/∂x = 0 |
| y = sigmoid(x) | ∂y/∂x = ∂y/∂out × y × (1-y) |

**遍历过程中，每个节点被访问时的动作**：

```
1. 接收从下游传来的梯度 ∂y/∂out（当前节点的输出对最终输出的偏导张量）
2. 用预置公式计算对每个输入的梯度（局部偏导张量）
3. 链式相乘：grad_in_i = grad_out × (∂out/∂input_i)
4. 将 grad_in_i 继续向上游传递
```

In [ ]:
# 验证 autograd 的计算结果与手动链式求导一致，并观察遍历顺序
import torch

# 设置数值：w 和 b 是需要求梯度的参数张量
# x 是输入数据，不需要梯度（默认 requires_grad=False）
x = torch.tensor([2.0])
w = torch.tensor([3.0], requires_grad=True)  # 参数1（标量张量）
b = torch.tensor([4.0], requires_grad=True)  # 参数2（标量张量）

# 前向传播
s = x + w          # s = 2 + 3 = 5
y = s * b          # y = 5 * 4 = 20

# autograd 自动求梯度：计算 ∂y/∂w 和 ∂y/∂b
y.backward()

print("=== autograd 计算的梯度 ===")
print(f"  ∂y/∂w = {w.grad.item()}")  # 4.0
print(f"  ∂y/∂b = {b.grad.item()}")  # 5.0

print("\n=== 手动链式求导验证 ===")
print(f"  ∂y/∂w = b = {b.item()}")  # 4.0
print(f"  ∂y/∂b = s = {s.item()}")  # 5.0

print("\n=== 每个参数张量的梯度 ===")
print(f"  w.grad 与 w 同 shape: {w.grad.shape}")  # torch.Size([])
print(f"  b.grad 与 b 同 shape: {b.grad.shape}")  # torch.Size([])
print("  每个参数张量各自有独立的 .grad，结构互不相同")

# 观测遍历顺序
print("\n=== 反向遍历顺序（DFS）===")
visited = []

def traverse(node, indent=0):
    """递归遍历计算图，观察访问顺序"""
    if node is None:
        return
    # 记录节点类型名称
    node_name = type(node).__name__
    visited.append(node_name)
    # 打印当前节点，缩进表示深度
    print("  " * indent + f"→ {node_name}")
    # 递归访问所有上游依赖
    for child, _ in node.next_functions:
        traverse(child, indent + 1)

traverse(y.grad_fn)
print(f"\n访问顺序：{' → '.join(visited)}")
# 输出示例：MulBackward0 → AddBackward0 → AccumulateGrad

print("\n遍历目的：从输出 y 出发，找到所有参数叶子（w 和 b），")
print("          计算每个参数张量的梯度（与参数同 shape），")
print("          只有这些参数张量的梯度需要保留，中间张量的梯度被丢弃")

## 4.7 多分支：同一个参数张量被多条路径"摸"到

当一个参数张量被多个分支共享时，从输出出发会有**多条路径**最终摸到同一个参数。

```
        分支1 → y1 ─┐
        /           │
      w             ├─→ loss = y1 + y2
        \           │
        分支2 → y2 ─┘
```

数学上：∂loss/∂w = ∂loss/∂y1 · ∂y1/∂w + ∂loss/∂y2 · ∂y2/∂w（全微分求导的链式法则）

autograd 的处理方式：两条路径都指向同一个 AccumulateGrad 节点，该节点将两条路径传来的梯度**累加**，再写入 w.grad。

In [ ]:
# 多分支共享参数：两条路径都摸到 w
import torch

# 唯一的参数张量 w，被两条分支共享
w = torch.tensor([2.0], requires_grad=True)

# 分支1：y1 = w * 3，∂y1/∂w = 3
y1 = w * 3

# 分支2：y2 = w * 5，∂y2/∂w = 5
y2 = w * 5

# loss = y1 + y2
loss = y1 + y2

# 反向传播
loss.backward()

# ∂loss/∂w = ∂loss/∂y1 * ∂y1/∂w + ∂loss/∂y2 * ∂y2/∂w
#          = 1 * 3 + 1 * 5 = 8
print(f"∂loss/∂w = {w.grad.item()}")  # 8.0
print("两条路径的梯度在 AccumulateGrad 处累加，不是覆盖")

## 4.8 为什么说"遍历"是关键机制？

**如果不遍历，autograd 无法工作**：

| 如果没有遍历 | 有了遍历 |
|------------|---------|
| 只知道输出 y，不知道还有哪些参数张量需要梯度 | 从 y 出发，沿 next_functions 找到所有参数叶子 |
| 无法知道哪些张量设置了 requires_grad=True | 遍历过程中 AccumulateGrad 标识出所有参数叶子 |
| 无法将梯度传递给正确的参数张量 | 遍历路径决定了 ∂loss/∂w 向谁传递 |

**一句话总结**：

> **遍历 = autograd 的"搜索算法"**——从最终输出出发，搜索整个计算图，找到所有需要梯度的参数张量，
  并在搜索过程中完成每个参数张量的梯度计算（与参数同 shape 的偏导张量）。

## 4.9 计算图生命周期：何时用完？为何销毁？

```
【前向阶段】autograd 唯一构图时机
├── 逐行执行张量运算
├── 判断 requires_grad 是否开启
├── 创建 XXXBackwardN 反向节点（保存前向中间值，如 s、b，用于反向求梯度）
├── 绑定 next_functions（记录上游依赖）
└── 赋值 grad_fn（让输出张量指向该节点）

【反向阶段】autograd 深度优先遍历，计算每个参数张量的梯度
├── 从输出张量的 grad_fn 入口进入
├── 深度优先遍历整张 DAG，追溯所有上游依赖
├── 依次调用各节点的内置求导函数（使用前向阶段保存的中间值）
├── 沿链路链式相乘，逐步得到每个参数张量的梯度
├── 多分支梯度在 AccumulateGrad 处累加
└── 将梯度写入所有参数叶子 .grad（与参数同 shape）

【销毁阶段】默认行为：一次 backward 完成后立即销毁
└── autograd 释放反向节点的循环引用，释放前向中间值占用的显存
```

**何时"用完"？**

- 一次 `backward()` 调用执行完毕，所有参数张量的 `.grad` 被填上梯度张量
- 每个参数张量的梯度与参数同 shape，可独立使用
- 此时所有反向节点的使命完成，计算图"用完"

**为何要销毁？**

| 原因 | 说明 |
|------|------|
| **释放显存** | 反向节点保存了前向运算的中间值（如 `s`、`b`），反向完成后不再需要。如果不释放，每个 batch 都会累积，显存爆炸 |
| **防止累积** | 训练中每步都构建新图，如果不销毁旧图，显存会持续增长直至 OOM |
| **性能优化** | 显存稳定是训练持续进行的前提，"用完即焚"是刻意设计 |

> **计算图是"一次性用品"**——设计上就是为了单次反向传播。autograd 默认"用完即焚"，以最小化显存占用。

**一个直观的理解**：

```
每个训练步骤：
1. 前向：构建新图（分配显存存放中间值）
2. 反向：遍历图求梯度（使用中间值计算每个参数张量的梯度）
3. 销毁：释放图（释放中间值占用的显存）
4. 下一轮：重复 1-3

如果不销毁 → 第1步的图还在，第2步又建新图 → 显存越积越多 → 崩溃
```

In [ ]:
# 演示 retain_graph=True 的作用
import torch

# 创建参数张量 w
w = torch.tensor([5.0], requires_grad=True)

# 前向运算：y = w^3
y = w ** 3

# 第1次反向：保留计算图
# retain_graph=True 告知 autograd：这张图我还要再用一次，不要销毁
y.backward(retain_graph=True)
print(f"第1次: ∂y/∂w = {w.grad.item()}")  # 75.0（3 * 5^2 = 75）

# 清零梯度，否则第2次会累加
w.grad.zero_()

# 第2次反向：复用原图，这次不加 retain_graph，默认销毁
y.backward()
print(f"第2次: ∂y/∂w = {w.grad.item()}")  # 75.0

# 第3次尝试：图已销毁，调用会报错
print("\n尝试第3次反向（图已销毁）...")
try:
    w.grad.zero_()
    y.backward()
except RuntimeError as e:
    print(f"报错（符合预期）: {e}")

print("\n结论：默认情况下，一次 backward 后图即销毁。")
print("      需多次反向时，使用 retain_graph=True 保留图。")

## 4.10 训练/推理场景 autograd 启停策略

| 场景 | autograd策略 | 原因 |
|------|-------------|------|
| 训练模式 | 全程开启追踪 | 需要构建计算图，计算每个参数张量的梯度更新权重 |
| 验证/测试 | with torch.no_grad(): | 只做前向推理，无需计算梯度，节省显存和加速 |
| 特征提取（冻结主干） | 主干用 detach() 截断 | 只训练特定分支，防止梯度影响冻结部分 |
| 梯度累积（大batch训练） | 多次 backward() + 单次 optimizer.step() | 手动控制autograd的多次反向时机 |

In [ ]:
# 典型推理写法：不需要梯度，不构建计算图
import torch
import torch.nn as nn

# 创建一个简单的线性模型
model = nn.Linear(10, 1)

# 随机输入数据
x = torch.randn(5, 10)

# 推理/验证时使用 torch.no_grad() 关闭 autograd 追踪
# 这样可以节省显存、加速计算
with torch.no_grad():
    out = model(x)

# 推理输出不需要梯度，也没有 grad_fn
print("推理输出 requires_grad:", out.requires_grad)  # False
print("推理输出 grad_fn:", out.grad_fn)  # None

# 附录一：计算图节点详解

## 1. 节点命名规则

```
XXXBackwardN
 │     │      │
 │     │      └── N：框架内部标识，用户无需关心
 │     └───────── Backward：框架内部标识，用户无需关心
 └─────────────── XXX：唯一对用户有意义的部分——一眼就知道这是什么运算
```

**对用户来说只需关注前缀**：

| 前缀 | 对应的前向运算 |
|------|--------------|
| `Add` | 加法 |
| `Mul` | 乘法 |
| `Relu` | ReLU激活 |
| `Sigmoid` | Sigmoid激活 |
| `Matmul` | 矩阵乘法 |
| `Mean` | 求均值 |
| `Pow` | 幂运算 |

## 2. 两类核心节点

| 节点类型 | 关联对象 | 生成时机 | 职责 |
|---------|---------|---------|------|
| **XXXBackwardN** | 中间张量 | 前向执行运算时**立即创建** | 保存该算子的求导逻辑；遍历时被autograd调用 |
| **AccumulateGrad** | 参数叶子张量（requires_grad=True） | **惰性创建** | 位于图末端；汇总所有流向该参数张量的梯度，求和后写入 .grad |

## 3. AccumulateGrad 的惰性创建

创建 `requires_grad=True` 的参数叶子张量时，autograd 仅登记身份（"准备"状态），**不立即创建 AccumulateGrad**。首次被反向节点引用时才真正实例化——这是 PyTorch 的性能优化策略。

```text
【时刻1】w = torch.tensor([3.0], requires_grad=True)
→ 标记 w 为"需要追踪"，但不创建 AccumulateGrad

【时刻2】s = x + w
→ 创建 AddBackward0，其 next_functions 指向 w
→ 此时 w 的 AccumulateGrad 才被真正创建！
```

> 参数叶子的 grad_fn 永远为 None——AccumulateGrad 存在于上游反向节点的 `next_functions` 中，不暴露给 `w.grad_fn`。

## 4. 高频误区

| 误区 | 真相 |
|------|------|
| AccumulateGrad 是 Tensor 的属性 | 是独立的反向节点，Tensor 通过上游节点的 next_functions 间接关联 |
| 所有张量都有 AccumulateGrad | 仅 requires_grad=True 的参数叶子才有 |
| 中间张量也有 AccumulateGrad | 永远不会，中间张量的梯度被计算后直接丢弃 |
| Tensor 有 next_functions | 只有反向节点实例才有，Tensor 没有 |
| requires_grad=True 时 AccumulateGrad 立即创建 | 惰性创建，首次被引用时才实例化 |

# 附录二：autograd 接口补充说明

## .backward() 的底层实现

`.backward()` 是 Tensor 暴露给用户的接口方法，其底层由 `torch.autograd.backward()` 实现——后者才是真正触发遍历求梯度的函数。

```python
# 这两行等价
y.backward()
torch.autograd.backward(y)
```

就像遥控器的按钮（Tensor方法）按下后，真正工作的是内部电路（autograd引擎）。

## torch.autograd 核心接口速查

| 接口 | 作用 |
|------|------|
| `.backward()` | 启动反向传播，计算每个参数张量的梯度 |
| `torch.no_grad()` | 临时关闭autograd追踪 |
| `.detach()` | 切断链路上梯度向上游的传播 |
| `retain_graph=True` | 保留计算图以供多次反向 |
| `.retain_grad()` | 强制保留中间张量的梯度 |
| `torch.autograd.backward()` | autograd 引擎的核心反向函数，触发图遍历 |
| `x.grad.zero_()` | 清零梯度（不关闭追踪） |

In [ ]:
# 演示 .backward() 和 torch.autograd.backward() 等价
import torch

print("=== 方式1：y.backward() ===")
x1 = torch.tensor([2.0], requires_grad=True)
y1 = x1 ** 2
y1.backward()
print(f"x1.grad = {x1.grad.item()}")  # 4.0

print("\n=== 方式2：torch.autograd.backward(y) ===")
x2 = torch.tensor([2.0], requires_grad=True)
y2 = x2 ** 2
torch.autograd.backward(y2)
print(f"x2.grad = {x2.grad.item()}")  # 4.0

print("\n结论：y.backward() 底层调用了 torch.autograd.backward(y)")
print("      .backward() 是 Tensor 的方法，autograd.backward() 是引擎的核心函数")

# 全文总结：autograd 核心要义

| 你所学到的 | 一句话总结 |
|-----------|-----------|
| autograd 是什么 | PyTorch 的自动微分引擎，专门自动化执行BP算法 |
| autograd 如何工作 | 拆分算子 → 预置导数 → 链式法则传递 |
| 计算图是什么 | autograd 在前向阶段自动构建的DAG，是执行BP的载体 |
| 计算图如何触发 | requires_grad=True 是"订阅信号"，**执行张量运算**才是真正触发构图的指令 |
| 图如何串联 | grad_fn 问"谁生了我"，next_functions 问"我依赖谁" |
| 反向传播做什么 | **深度优先遍历计算图**：从最终输出 y 出发，沿 next_functions 追溯，找到所有参数张量，计算每个参数张量的梯度（与参数同 shape） |
| 数学本质 | 多元函数全微分的链式求导法则；梯度是与参数张量同 shape 的偏导张量 |
| 每个节点做什么 | 接收 ∂y/∂out → 用预置公式求局部梯度 → 链式相乘得到 ∂y/∂in → 向上游传递 |
| 遍历顺序 | 深度优先（DFS），由链式法则的依赖关系决定 |
| 梯度需要保留在谁身上？ | **模型参数（叶子张量）**：永久保留，供优化器使用；**中间张量**：默认丢弃，只用做传递 |
| 多分支如何处理 | 同一参数张量的多条路径在 AccumulateGrad 处累加 |
| 计算图何时销毁 | 一次 backward() 完成后默认立即销毁，释放显存 |
| 如何保留图 | `retain_graph=True` 告知引擎"这张图我还要再用一次" |
| 如何控制 autograd | requires_grad（总开关）、detach/no_grad（切断追踪）、retain_graph（生命周期干预） |
| autograd 的核心价值 | 让任意复杂网络 ∂loss/∂w 的计算全自动化，支撑深度学习规模化 |

**最终认知**：

Tensor 是 PyTorch 的"肌肉"——承载数值与状态的基础数据结构。
autograd 是 PyTorch 的"神经系统"——赋予 Tensor "自动微分"能力。

**反向传播的本质**：

```
训练目标：更新模型参数（所有 nn.Parameter，即 requires_grad=True 的叶子张量）
数学目标：计算每个参数张量 w 的梯度：w.grad[i] = ∂loss/∂w[i]（与 w 同 shape）
反向传播：遍历计算图，计算每个参数张量的梯度张量
中间张量：梯度传递的"桥梁"，用过即弃（默认丢弃）
最终结果：每个参数张量的 .grad 存储对应的梯度张量，与参数同 shape
```

反向传播的本质就是**对计算图的一次深度优先遍历**：
- **遍历对象**：计算图（由 XXXBackwardN 和 AccumulateGrad 节点组成的 DAG）
- **遍历方向**：从最终输出 y 到参数叶子 w_i（与数据流方向相反）
- **遍历顺序**：深度优先（DFS），由链式法则的依赖关系决定
- **遍历目的**：找到所有 requires_grad=True 的参数张量，计算每个参数的梯度
- **遍历工具**：grad_fn（入口）+ next_functions（前进方向）
- **遍历结果**：每个参数张量的 .grad 被填上对应的梯度张量（与参数同 shape）
- **生命周期**：一次 backward 后默认销毁，释放显存；需多次反向时用 retain_graph=True 保留

深度学习能跑起来，靠的是神经系统（autograd）指挥肌肉（Tensor）完成每一次前向和反向的精密配合。